# Auditoria piloto de AIME

Objetivo: comprobar acceso, configuracion, splits, numero de filas, columnas y features de `disco-eth/AIME` sin descargar ni decodificar audio.

In [1]:
import os
import sys
from pathlib import Path
from importlib import metadata

# Cache local ignorada por git; se fija antes de importar librerias de Hugging Face.
cwd = Path.cwd()
repo_root = cwd if (cwd / ".git").exists() else cwd.parent
os.environ.setdefault("HF_HOME", str(repo_root / ".cache" / "huggingface"))

import datasets
from huggingface_hub import HfApi
import numpy as np
import pandas as pd

packages = ["datasets", "huggingface_hub", "numpy", "pandas", "torch", "torchaudio", "librosa"]
versions = {
    "python": sys.version.replace("\n", " "),
    "python_executable": sys.executable,
}

for package in packages:
    try:
        versions[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        versions[package] = "NOT INSTALLED"

versions

c:\Users\sergio\tfm\tfm-ai-music-detection\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'python': '3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]',
 'python_executable': 'c:\\Users\\sergio\\tfm\\tfm-ai-music-detection\\.venv\\Scripts\\python.exe',
 'datasets': '5.0.0',
 'huggingface_hub': '1.21.0',
 'numpy': '2.4.6',
 'pandas': '3.0.3',
 'torch': 'NOT INSTALLED',
 'torchaudio': 'NOT INSTALLED',
 'librosa': '0.11.0'}

## 1. Acceso y esquema

In [2]:
dataset_id = "disco-eth/AIME"

api = HfApi()
repo_info = api.dataset_info(dataset_id, files_metadata=False)

hub_summary = {
    "id": repo_info.id,
    "sha": repo_info.sha,
    "siblings_count": len(repo_info.siblings or []),
    "first_files": [s.rfilename for s in (repo_info.siblings or [])[:8]],
    "card_data_type": type(repo_info.cardData).__name__,
}

hub_summary

{'id': 'disco-eth/AIME',
 'sha': 'b84d4be5eda830b6eb714998569dba73530f2601',
 'siblings_count': 212,
 'first_files': ['.gitattributes',
  'README.md',
  'data/train-00000-of-00210.parquet',
  'data/train-00001-of-00210.parquet',
  'data/train-00002-of-00210.parquet',
  'data/train-00003-of-00210.parquet',
  'data/train-00004-of-00210.parquet',
  'data/train-00005-of-00210.parquet'],
 'card_data_type': 'DatasetCardData'}

In [3]:
datasets_api = {
    "version": datasets.__version__,
    "has_get_dataset_config_names": hasattr(datasets, "get_dataset_config_names"),
    "has_get_dataset_split_names": hasattr(datasets, "get_dataset_split_names"),
    "has_get_dataset_infos": hasattr(datasets, "get_dataset_infos"),
    "has_get_dataset_config_info": hasattr(datasets, "get_dataset_config_info"),
}

configs = datasets.get_dataset_config_names(dataset_id)
splits = datasets.get_dataset_split_names(dataset_id)
dataset_info = datasets.get_dataset_config_info(dataset_id, config_name="default")

columns = list(dataset_info.features.keys())
features = {name: repr(feature) for name, feature in dataset_info.features.items()}
split_rows = {name: split.num_examples for name, split in dataset_info.splits.items()}
split_num_bytes = {name: split.num_bytes for name, split in dataset_info.splits.items()}

schema_summary = {
    "configs": configs,
    "splits": splits,
    "split_rows": split_rows,
    "columns": columns,
    "features": features,
    "split_num_bytes": split_num_bytes,
    "download_size": dataset_info.download_size,
    "dataset_size": dataset_info.dataset_size,
}

datasets_api, schema_summary

({'version': '5.0.0',
  'has_get_dataset_config_names': True,
  'has_get_dataset_split_names': True,
  'has_get_dataset_infos': True,
  'has_get_dataset_config_info': True},
 {'configs': ['default'],
  'splits': ['train'],
  'split_rows': {'train': 6500},
  'columns': ['id', 'model', 'description', 'audio'],
  'features': {'id': "Value('string')",
   'model': "Value('string')",
   'description': "Value('string')",
   'audio': 'Audio(sampling_rate=None, decode=True, num_channels=None, stream_index=None)'},
  'split_num_bytes': {'train': 62747721096.5},
  'download_size': 62281475287,
  'dataset_size': 62747721096.5})

In [4]:
observed = {
    "method": "HfApi.dataset_info + datasets.get_dataset_config_names/get_dataset_split_names/get_dataset_config_info",
    "audio_downloaded_or_decoded": False,
    "dataset_id": dataset_id,
    "repo_sha": hub_summary["sha"],
    "configs": configs,
    "splits": splits,
    "rows": split_rows,
    "columns": columns,
    "features": features,
    "warnings": [
        "No se ha llamado a load_dataset ni se ha accedido a ejemplos o a la columna audio.",
        "La feature audio aparece con decode=True por defecto; acceder a ejemplos podria descargar o decodificar audio.",
    ],
}

print("RESULTADOS OBSERVADOS - FASE 1")
for key, value in observed.items():
    print(f"{key}: {value}")

RESULTADOS OBSERVADOS - FASE 1
method: HfApi.dataset_info + datasets.get_dataset_config_names/get_dataset_split_names/get_dataset_config_info
audio_downloaded_or_decoded: False
dataset_id: disco-eth/AIME
repo_sha: b84d4be5eda830b6eb714998569dba73530f2601
configs: ['default']
splits: ['train']
rows: {'train': 6500}
columns: ['id', 'model', 'description', 'audio']
features: {'id': "Value('string')", 'model': "Value('string')", 'description': "Value('string')", 'audio': 'Audio(sampling_rate=None, decode=True, num_channels=None, stream_index=None)'}
warnings: ['No se ha llamado a load_dataset ni se ha accedido a ejemplos o a la columna audio.', 'La feature audio aparece con decode=True por defecto; acceder a ejemplos podria descargar o decodificar audio.']
